In [ ]:
# Referenced from: https://www.kaggle.com/code/ankushnarwade/connectx-agent-using-q-learning-dqn/notebook

# 1. Install kaggle environments
!pip install kaggle-environments

import numpy as np
import random
import base64
import os
from tqdm import tqdm

from collections import deque
from kaggle_environments import make

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Input, Lambda
from tensorflow.keras.optimizers import Adam

In [ ]:
# 1. Re-initialize the environment and config here to guarantee they exist
env = make("connectx", debug=False)
config = env.configuration

In [ ]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.memory = []
        self.priorities = deque(maxlen=capacity)

    def add(self, experience, error=1.0):
        priority = (abs(error) + 1e-5) ** self.alpha
        if len(self.memory) < self.capacity:
            self.memory.append(experience)
            self.priorities.append(priority)
        else:
            self.memory.pop(0)
            self.memory.append(experience)
            self.priorities.append(priority)

    def sample(self, batch_size, beta=0.4):
        priorities = np.array(self.priorities)
        probs = priorities / priorities.sum()

        indices = np.random.choice(len(self.memory), batch_size, p=probs)
        samples = [self.memory[i] for i in indices]

        weights = (len(self.memory) * probs[indices]) ** (-beta)
        weights /= weights.max()

        return samples, indices, weights

    def update_priorities(self, indices, errors):
        for i, e in zip(indices, errors):
            self.priorities[i] = (abs(e) + 1e-5) ** self.alpha

In [ ]:
class DQNAgent:
    def __init__(self, rows, columns, action_size):
        self.rows = rows
        self.columns = columns
        self.action_size = action_size

        self.memory = PrioritizedReplayBuffer(20000)

        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995

        self.beta = 0.4
        self.beta_increment = 0.001

        self.model = self.build_model()
        self.target_model = self.build_model()
        self.update_target_network()

        self.train_step = 0

    def build_model(self):
        inputs = Input(shape=(self.rows, self.columns, 1))

        x = Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
        x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
        x = Flatten()(x)

        # Dueling
        v = Dense(64, activation='relu')(x)
        v = Dense(1)(v)

        a = Dense(64, activation='relu')(x)
        a = Dense(self.action_size)(a)

        q = Lambda(lambda z: z[0] + (z[1] - tf.reduce_mean(z[1], axis=1, keepdims=True)))([v, a])

        model = Model(inputs, q)
        model.compile(loss='mse', optimizer=Adam(0.001))
        return model

    def update_target_network(self):
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, *args):
        self.memory.add(args)

    def act(self, state, valid_moves):
        if np.random.rand() < self.epsilon:
            return random.choice(valid_moves)

        q = self.model.predict(state[np.newaxis], verbose=0)[0]

        masked = np.full_like(q, -1e9)
        masked[valid_moves] = q[valid_moves]

        return int(np.argmax(masked))

    def replay(self, batch_size=64):
        if len(self.memory.memory) < batch_size:
            return

        batch, idxs, weights = self.memory.sample(batch_size, self.beta)

        states = np.array([b[0] for b in batch])
        actions = np.array([b[1] for b in batch])
        rewards = np.array([b[2] for b in batch])
        next_states = np.array([b[3] for b in batch])
        dones = np.array([b[4] for b in batch])

        q = self.model.predict(states, verbose=0)
        next_main = self.model.predict(next_states, verbose=0)
        next_target = self.target_model.predict(next_states, verbose=0)

        best = np.argmax(next_main, axis=1)
        target_q = next_target[np.arange(batch_size), best]

        targets = q.copy()
        td_errors = []

        for i in range(batch_size):
            t = rewards[i] if dones[i] else rewards[i] + self.gamma * target_q[i]
            td_errors.append(t - q[i, actions[i]])
            targets[i, actions[i]] = t

        self.model.fit(states, targets, sample_weight=weights, verbose=0)

        self.memory.update_priorities(idxs, td_errors)
        self.beta = min(1.0, self.beta + self.beta_increment)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        self.train_step += 1
        if self.train_step % 200 == 0:
            self.update_target_network()

In [ ]:
# --- Helper Functions ---
def get_valid_moves(board, config):
    return [c for c in range(config.columns) if board[c] == 0]

def state_to_np(board, config, mark):
    grid = np.array(board).reshape(config.rows, config.columns)

    # Normalize: self = 1, opponent = -1
    grid = np.where(grid == mark, 1, grid)
    grid = np.where(grid == (1 if mark == 2 else 2), -1, grid)

    return grid.reshape(config.rows, config.columns, 1).astype(np.float32)

def mirror_board(board, config):
    grid = np.array(board).reshape(config.rows, config.columns)
    return np.fliplr(grid).flatten()

def mirror_action(action, config):
    return config.columns - 1 - action

def check_window(window, num_discs, piece, config):
    # Checks if a slice of 4 slots contains exactly 'num_discs' of our piece, and the rest are empty
    return (window.count(piece) == num_discs and window.count(0) == config.inarow - num_discs)

def count_windows(grid, num_discs, piece, config):
    num_windows = 0
    # Convert grid to 2D array for easier slicing
    grid_2d = np.array(grid).reshape(config.rows, config.columns)
    
    # Horizontal
    for r in range(config.rows):
        for c in range(config.columns - 3):
            window = list(grid_2d[r, c:c+4])
            if check_window(window, num_discs, piece, config): num_windows += 1
            
    # Vertical
    for r in range(config.rows - 3):
        for c in range(config.columns):
            window = list(grid_2d[r:r+4, c])
            if check_window(window, num_discs, piece, config): num_windows += 1
            
    # Positive Diagonal
    for r in range(config.rows - 3):
        for c in range(config.columns - 3):
            window = list(grid_2d[r:r+4, c:c+4].diagonal())
            if check_window(window, num_discs, piece, config): num_windows += 1
            
    # Negative Diagonal
    for r in range(3, config.rows):
        for c in range(config.columns - 3):
            window = list(np.fliplr(grid_2d[r-3:r+1, c:c+4]).diagonal())
            if check_window(window, num_discs, piece, config): num_windows += 1
            
    return num_windows

def get_shaped_reward(prev_board, board, action, step_reward, config, done, mark):
    rows = config.rows
    cols = config.columns
    opp = 1 if mark == 2 else 2

    # 1. Terminal rewards (unchanged)
    if done:
        if step_reward == 1:
            return 100
        elif step_reward == -1:
            return -100
        else:
            return 0

    reward = 0

    # =========================
    # 2. COUNT BEFORE / AFTER
    # =========================
    prev_my_3 = count_windows(prev_board, 3, mark, config)
    prev_my_2 = count_windows(prev_board, 2, mark, config)
    prev_opp_3 = count_windows(prev_board, 3, opp, config)

    new_my_3 = count_windows(board, 3, mark, config)
    new_my_2 = count_windows(board, 2, mark, config)
    new_opp_3 = count_windows(board, 3, opp, config)

    # =========================
    # 3. DELTA REWARD (KEY)
    # =========================
    reward += (new_my_3 - prev_my_3) * 8
    reward += (new_my_2 - prev_my_2) * 2

    reward -= (new_opp_3 - prev_opp_3) * 12

    # =========================
    # 4. CENTER CONTROL (after state only)
    # =========================
    grid = np.array(board).reshape(rows, cols)
    center_col = cols // 2
    center_count = list(grid[:, center_col]).count(mark)
    reward += center_count * 2

    # =========================
    # 5. ANTI-BLUNDER
    # =========================
    if prev_opp_3 > 0 and new_opp_3 > prev_opp_3:
        reward -= 15  # made opponent stronger

    return reward

In [ ]:
from tqdm import tqdm
from collections import deque

env = make("connectx", debug=False)

agent = DQNAgent(config.rows, config.columns, config.columns)

episodes = 10000
step = 0

reward_history = deque(maxlen=100)
win_history = deque(maxlen=100)

def check_win(board, mark):
    grid = np.array(board).reshape(config.rows, config.columns)
    for r in range(config.rows):
        for c in range(config.columns-3):
            if all(grid[r, c+i] == mark for i in range(4)): return True
    for c in range(config.columns):
        for r in range(config.rows-3):
            if all(grid[r+i, c] == mark for i in range(4)): return True
    for r in range(config.rows-3):
        for c in range(config.columns-3):
            if all(grid[r+i, c+i] == mark for i in range(4)): return True
    for r in range(3, config.rows):
        for c in range(config.columns-3):
            if all(grid[r-i, c+i] == mark for i in range(4)): return True
    return False

pbar = tqdm(range(episodes), desc="Self-Play Training")

for ep in pbar:
    env.reset()
    board = env.state[0]["observation"]["board"]

    done = False
    current_player = 1
    total_reward = 0

    while not done:
        state = state_to_np(board, config, current_player)
        valid_moves = [c for c in range(config.columns) if board[c] == 0]

        action = agent.act(state, valid_moves)

        # exploration
        if np.random.rand() < max(0.05, agent.epsilon * 0.5):
            action = random.choice(valid_moves)

        # apply move
        next_board = list(board)
        for r in range(config.rows-1, -1, -1):
            idx = r * config.columns + action
            if next_board[idx] == 0:
                next_board[idx] = current_player
                break

        # outcome
        win = check_win(next_board, current_player)
        draw = all(x != 0 for x in next_board)

        if win:
            reward = 1.0
            done = True
            winner = current_player

        elif draw:
            reward = 0.0
            done = True
            winner = 0

        else:
            reward = -0.01   # mild shaping
            done = False
            winner = None

        next_state = state_to_np(next_board, config, current_player)

        # store experience
        agent.remember(state, action, reward, next_state, done)

        # symmetry augmentation
        mb = mirror_board(board, config)
        mn = mirror_board(next_board, config)

        agent.remember(
            state_to_np(mb, config, current_player),
            mirror_action(action, config),
            reward,
            state_to_np(mn, config, current_player),
            done
        )

        board = next_board
        current_player = 1 if current_player == 2 else 2
        total_reward += reward
        step += 1

        # more stable training
        if len(agent.memory.memory) > 5000 and step % 20 == 0:
            agent.replay()

    # target network update
    if ep % 20 == 0:
        agent.update_target_network()

    reward_history.append(total_reward)

    # win tracking
    if winner == 1:
        win_history.append(1)
    elif winner == 2:
        win_history.append(0)
    else:
        win_history.append(0.5)

    pbar.set_postfix({
        "avg_reward": round(np.mean(reward_history), 3),
        "win_rate": round(np.mean(win_history), 3),
        "epsilon": round(agent.epsilon, 3),
        "memory": len(agent.memory.memory)
    })

In [ ]:
# --- Save and Encode Model ---
agent.model.save("connectx_dqn.h5")
print("Model saved to notebook environment!")

with open("connectx_dqn.h5", "rb") as f:
    model_bytes = f.read()
encoded_model_string = base64.b64encode(model_bytes).decode('utf-8')

In [ ]:
agent_code = f"""
import os
import base64
import numpy as np
from tensorflow.keras.models import load_model
import tensorflow as tf

# --- Load model safely (handles Lambda layer) ---
encoded_model = "{encoded_model_string}"
model_path = "/tmp/connectx_model.h5"

if not os.path.exists(model_path):
    with open(model_path, "wb") as f:
        f.write(base64.b64decode(encoded_model))

def dueling_combine(v, a):
    return v + (a - tf.reduce_mean(a, axis=1, keepdims=True))

model = load_model(model_path, compile=False)

# --- Helpers ---
def simulate_drop(board, col, mark, config):
    b = list(board)
    for r in range(config.rows - 1, -1, -1):
        idx = r * config.columns + col
        if b[idx] == 0:
            b[idx] = mark
            break
    return b

def check_win(board, mark, config):
    grid = np.array(board).reshape(config.rows, config.columns)

    for r in range(config.rows):
        for c in range(config.columns - 3):
            if all(grid[r, c+i] == mark for i in range(4)):
                return True

    for c in range(config.columns):
        for r in range(config.rows - 3):
            if all(grid[r+i, c] == mark for i in range(4)):
                return True

    for r in range(config.rows - 3):
        for c in range(config.columns - 3):
            if all(grid[r+i, c+i] == mark for i in range(4)):
                return True

    for r in range(3, config.rows):
        for c in range(config.columns - 3):
            if all(grid[r-i, c+i] == mark for i in range(4)):
                return True

    return False

# Normalize board (VERY IMPORTANT)
def encode(board, mark, config):
    grid = np.array(board).reshape(config.rows, config.columns)

    grid = np.where(grid == mark, 1, grid)
    grid = np.where(grid == (1 if mark == 2 else 2), -1, grid)

    return grid.reshape(1, config.rows, config.columns, 1).astype(np.float32)

# Mirror utilities
def mirror_board(board, config):
    grid = np.array(board).reshape(config.rows, config.columns)
    return np.fliplr(grid).flatten()

def mirror_action(action, config):
    return config.columns - 1 - action

# --- Main Agent ---
def my_agent(observation, configuration):
    board = observation.board
    mark = observation.mark
    opp = 1 if mark == 2 else 2

    valid = [c for c in range(configuration.columns) if board[c] == 0]
    if not valid:
        return 0

    # 1. WIN
    for c in valid:
        if check_win(simulate_drop(board, c, mark, configuration), mark, configuration):
            return c

    # 2. BLOCK
    for c in valid:
        if check_win(simulate_drop(board, c, opp, configuration), opp, configuration):
            return c

    # 3. CENTER PRIORITY
    center = configuration.columns // 2
    if center in valid:
        return center

    # 4. MODEL (normal)
    state = encode(board, mark, configuration)
    q = model.predict(state, verbose=0)[0]

    # 5. MODEL (mirror for stability)
    mirrored = mirror_board(board, configuration)
    state_m = encode(mirrored, mark, configuration)
    q_m = model.predict(state_m, verbose=0)[0]

    # un-mirror q-values
    q_m = np.array([q_m[mirror_action(i, configuration)] for i in range(configuration.columns)])

    # Ensemble (very strong)
    center = configuration.columns // 2
    q[center] += 0.5

    # Mask invalid
    masked = np.full_like(q, -1e9)
    masked[valid] = q[valid]

    action = int(np.argmax(masked))

    return action
"""

with open("submission.py", "w") as f:
    f.write(agent_code)

print("submission.py saved!")

In [ ]:
# --- Test Agent in Notebook ---
# Evaluate the newly created python file against kaggle's negamax
env.reset()
env.run(["submission.py", "negamax"])
env.render(mode="ipython", width=500, height=450)